In [0]:

df = spark.read.csv("/Volumes/filestore/olist/data/C1.csv",header=True,inferSchema=False  )

# Remove duplicates
df = df.dropDuplicates()

# Clean Country
df = df.withColumn(
    "Country",
    when(lower(col("Country")) == "new york", "USA")
    .otherwise(initcap(col("Country")))
)

# Remove blanks
df = df.filter(
    (col("Sales") != "blank") &
    (col("Category") != "Blank") &
    (col("JoinDate") != "blank")
)

# ✅ Fix date (NO timestamp → use DATE only)
df = df.withColumn(
    "JoinDate",
    coalesce(
        expr("try_to_date(JoinDate, 'dd-MM-yyyy')"),
        expr("try_to_date(JoinDate, 'MM-dd-yyyy')")
    )
)

# Format date
df = df.withColumn("JoinDate", date_format(col("JoinDate"), "dd-MM-yyyy"))

# Convert Sales
df = df.withColumn("Sales", col("Sales").cast("int"))

# Final output
df = df.orderBy(col("CustomerID").asc())

df.show()

+----------+-------+-------+----------+-----+--------+
|CustomerID|   Name|Country|  JoinDate|Sales|Category|
+----------+-------+-------+----------+-----+--------+
|       101|  Alice|    Usa|15-01-2023|  250|       A|
|       102|    Bob|  India|01-05-2023|  150|       B|
|       104|  Alice|    Usa|15-01-2023|  250|       A|
|       106|Mallory|    USA|15-03-2023|  400|       B|
|       107|  Trent|  India|10-04-2023|  350|       B|
|       108|    Bob|  India|05-01-2023|  150|       B|
|       109|  Oscar|    Usa|28-02-2023|  500|       A|
+----------+-------+-------+----------+-----+--------+

